# Pronóstico de una serie temporal con `statsmodels`

## Introducción

El **pronóstico** consiste en estimar valores futuros a partir del comportamiento histórico de una variable. Se utiliza para apoyar decisiones como planear inventarios, personal, capacidad operativa, ventas o presupuesto.

En este ejercicio analizaremos el número mensual de pasajeros de una aerolínea. El problema que buscamos resolver es: **¿cuántos pasajeros podríamos esperar durante los próximos meses, considerando el crecimiento general y los patrones que se repiten cada año?**

La variable tiene dos características importantes: una tendencia de crecimiento y una estacionalidad anual. Por ello, compararemos una línea base sencilla contra un modelo Holt-Winters de la librería `statsmodels`.

## Objetivos del ejercicio

Al finalizar podremos:

1. Reconocer tendencia y estacionalidad en una serie temporal.
2. Separar correctamente los datos históricos de los datos futuros.
3. Crear un pronóstico base y un pronóstico con Holt-Winters.
4. Medir el error de los pronósticos.
5. Interpretar los resultados y sus limitaciones.

## Configuración para Google Colab

Google Colab suele incluir estas librerías, pero esta celda asegura que el notebook utilice versiones disponibles de `statsmodels` y sus dependencias. Ejecútela solamente si el entorno solicita instalar o actualizar paquetes.

In [ ]:
%pip install -q statsmodels seaborn scikit-learn

## 1. Preparar el entorno

Este bloque importa las librerías. `pandas` organiza los datos, `matplotlib` y `seaborn` generan gráficas, `statsmodels` contiene el modelo de pronóstico y `sklearn` permite calcular métricas de error.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

## 2. Cargar el dataset

Usaremos el dataset `flights` de `seaborn`, que contiene pasajeros mensuales de 1949 a 1960. La carga principal utiliza una librería popular y activa. El bloque incluye un respaldo sintético para que el notebook no se detenga si el entorno no tiene acceso a Internet.

In [ ]:
try:
    vuelos = sns.load_dataset('flights')
    fuente_datos = 'Dataset flights de seaborn'
except Exception:
    # Respaldo reproducible: serie con tendencia y patrón anual
    fechas = pd.date_range('2010-01-01', periods=144, freq='MS')
    rng = np.random.default_rng(42)
    estacionalidad = np.array([0.82, 0.78, 0.92, 0.98, 1.08, 1.18, 1.28, 1.23, 1.08, 0.96, 0.86, 0.80])
    tendencia = np.linspace(180, 360, len(fechas))
    pasajeros = tendencia * np.tile(estacionalidad, 12) + rng.normal(0, 5, len(fechas))
    vuelos = pd.DataFrame({'year': fechas.year, 'month': fechas.strftime('%b'), 'passengers': pasajeros.round().astype(int)})
    fuente_datos = 'Dataset sintético de respaldo'

vuelos.head()

## 3. Preparar la serie temporal

El modelo necesita una fecha como índice y una observación numérica por periodo. Convertiremos año y mes en una fecha mensual, ordenaremos los registros y revisaremos la cantidad de observaciones. Esta validación evita errores comunes antes de modelar.

In [ ]:
vuelos['fecha'] = pd.to_datetime(vuelos['year'].astype(str) + '-' + vuelos['month'].astype(str) + '-01')
serie = vuelos.set_index('fecha')['passengers'].sort_index().asfreq('MS')

print(f'Fuente: {fuente_datos}')
print(f'Periodo: {serie.index.min():%Y-%m} a {serie.index.max():%Y-%m}')
print(f'Observaciones: {len(serie)}')
print(f'Valores faltantes: {serie.isna().sum()}')
serie.head()

## 4. Explorar tendencia y estacionalidad

La gráfica ayuda a responder si la demanda crece con el tiempo y si ciertos meses suelen presentar niveles distintos. Identificar estos patrones orienta la elección del modelo: Holt-Winters es apropiado cuando existen nivel, tendencia y estacionalidad.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
serie.plot(ax=axes[0], color='#2563eb', linewidth=2)
axes[0].set_title('Pasajeros mensuales: tendencia histórica')
axes[0].set_ylabel('Pasajeros')

promedio_mensual = serie.groupby(serie.index.month).mean()
promedio_mensual.index = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
promedio_mensual.plot(kind='bar', ax=axes[1], color='#f59e0b')
axes[1].set_title('Promedio histórico por mes: patrón estacional')
axes[1].set_ylabel('Pasajeros promedio')
axes[1].set_xlabel('Mes')
plt.tight_layout()

## 5. Separar entrenamiento y prueba

En pronóstico no debemos revolver aleatoriamente los registros, porque eso permitiría que el modelo aprendiera del futuro. Usaremos los últimos 12 meses como prueba y todos los meses anteriores para entrenar. Así simulamos una situación real: pronosticamos un periodo que todavía no conocemos.

In [ ]:
horizonte_prueba = 12
entrenamiento = serie.iloc[:-horizonte_prueba]
prueba = serie.iloc[-horizonte_prueba:]

print(f'Entrenamiento: {entrenamiento.index.min():%Y-%m} a {entrenamiento.index.max():%Y-%m}')
print(f'Prueba: {prueba.index.min():%Y-%m} a {prueba.index.max():%Y-%m}')

## 6. Crear una línea base

Antes de usar un modelo sofisticado necesitamos una referencia sencilla. La línea base repite el último valor observado durante todo el horizonte de prueba. Un modelo útil debería superar esta referencia; de lo contrario, su complejidad no está aportando valor.

In [ ]:
pronostico_base = pd.Series(entrenamiento.iloc[-1], index=prueba.index, name='Base')
pronostico_base.head()

## 7. Entrenar Holt-Winters con `statsmodels`

Holt-Winters estima el nivel actual, la tendencia y el comportamiento estacional. `trend='add'` modela un crecimiento aproximadamente constante; `seasonal='mul'` permite que las variaciones estacionales crezcan junto con el nivel de pasajeros; `seasonal_periods=12` indica que el patrón se repite cada 12 meses.

In [ ]:
modelo = ExponentialSmoothing(
    entrenamiento,
    trend='add',
    seasonal='mul',
    seasonal_periods=12,
    initialization_method='estimated'
).fit(optimized=True)

pronostico_hw = modelo.forecast(horizonte_prueba)
pronostico_hw.name = 'Holt-Winters'
pronostico_hw.head()

## 8. Evaluar los pronósticos

Las métricas comparan cada pronóstico con los valores reales del periodo de prueba. MAE representa el error absoluto promedio en pasajeros; RMSE penaliza más los errores grandes; MAPE expresa el error promedio como porcentaje. En las tres métricas, un valor menor es mejor.

In [ ]:
def calcular_metricas(real, pronostico):
    mae = mean_absolute_error(real, pronostico)
    rmse = np.sqrt(mean_squared_error(real, pronostico))
    mape = np.mean(np.abs((real - pronostico) / real)) * 100
    return pd.Series({'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape})

metricas = pd.DataFrame({
    'Pronóstico base': calcular_metricas(prueba, pronostico_base),
    'Holt-Winters': calcular_metricas(prueba, pronostico_hw)
}).T
metricas

## 9. Interpretar visualmente el desempeño

La gráfica permite verificar si el pronóstico sigue el nivel general y los picos estacionales. No basta con observar una métrica: también conviene revisar si los errores se concentran en meses específicos o si el modelo pierde el crecimiento de la serie.

In [ ]:
plt.figure(figsize=(13, 6))
plt.plot(entrenamiento.index, entrenamiento, label='Entrenamiento', color='#64748b')
plt.plot(prueba.index, prueba, label='Real', color='#111827', linewidth=2)
plt.plot(pronostico_base.index, pronostico_base, '--', label='Base', color='#ef4444')
plt.plot(pronostico_hw.index, pronostico_hw, '--', label='Holt-Winters', color='#16a34a', linewidth=2)
plt.axvline(prueba.index[0], color='black', linestyle=':', label='Inicio de prueba')
plt.title('Comparación de pronósticos contra valores reales')
plt.ylabel('Pasajeros')
plt.xlabel('Fecha')
plt.legend()
plt.tight_layout()

## 10. Interpretación automática de resultados

Este bloque traduce la tabla de métricas a una lectura de negocio. La conclusión se basa en el MAPE de Holt-Winters y en su comparación contra la línea base, evitando afirmar que un modelo es bueno sin medirlo.

In [ ]:
mape_base = metricas.loc['Pronóstico base', 'MAPE (%)']
mape_hw = metricas.loc['Holt-Winters', 'MAPE (%)']
mejora = (1 - mape_hw / mape_base) * 100

print(f'Error porcentual de la línea base: {mape_base:.2f}%')
print(f'Error porcentual de Holt-Winters: {mape_hw:.2f}%')
if mejora > 0:
    print(f'Interpretación: Holt-Winters reduce el MAPE en {mejora:.2f}% frente a la línea base.')
else:
    print(f'Interpretación: Holt-Winters no supera la línea base en este periodo; la mejora es {mejora:.2f}%.')
print('Los errores deben interpretarse en el contexto del volumen de pasajeros y del costo de equivocarse.')

## 11. Pronóstico de los siguientes 12 meses

Después de evaluar el modelo, lo reentrenamos con toda la historia disponible. Esto permite aprovechar también los últimos 12 meses antes de generar el pronóstico futuro. La tabla final muestra los valores estimados para apoyar la planeación.

In [ ]:
modelo_final = ExponentialSmoothing(
    serie, trend='add', seasonal='mul', seasonal_periods=12,
    initialization_method='estimated'
).fit(optimized=True)

pronostico_futuro = modelo_final.forecast(12)
pronostico_futuro = pronostico_futuro.rename('Pasajeros pronosticados').round(0).astype(int)
pronostico_futuro.to_frame()

## Interpretación del pronóstico futuro

El resultado debe leerse como una estimación, no como una certeza. Los meses con valores mayores representan periodos de demanda esperada más alta. La utilidad práctica dependerá de actualizar el modelo cuando lleguen nuevos datos, monitorear el error y revisar cambios externos como precios, capacidad, competencia o eventos extraordinarios.

## Conclusiones

- Un pronóstico comienza con entender la serie temporal y sus patrones, no con elegir el modelo más complejo.
- La separación temporal entre entrenamiento y prueba es esencial para medir el desempeño de forma realista.
- La línea base permite comprobar si el modelo realmente agrega valor.
- Holt-Winters es apropiado para este ejemplo porque representa nivel, tendencia y estacionalidad anual.
- Las métricas indican qué tan cerca estuvieron las predicciones, pero la decisión final debe considerar el impacto operativo del error.
- En un proyecto real convendría comparar otros modelos, realizar validación temporal repetida, incorporar variables externas y monitorear el pronóstico continuamente.